# Notebook 02 — Multimodal Fusion Encoder

## Goal

Build a reusable multimodal encoder that projects MRI image features and
radiology report embeddings into a shared latent representation.

The generated representation will later be used by downstream clinical
decision support tasks.

# Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# Configuration

In [2]:
FUSION_DIR = Path(
    "/kaggle/input/datasets/mariammohamed1095/models/datasets/processed/fusion/clinicalbert"
)

BATCH_SIZE = 16

In [3]:
device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print(device)

cpu


# Load Fusion Dataset

In [4]:
def load_split(split):

    data = np.load(
        FUSION_DIR / f"{split}_fusion.npz"
    )

    metadata = pd.read_csv(
        FUSION_DIR / f"{split}_metadata.csv"
    )

    return {

        "image_features": data["image_features"],

        "text_features": data["text_features"],

        "metadata": metadata,

    }

In [5]:
train = load_split("train")

validation = load_split("validation")

test = load_split("test")

In [6]:
for name, data in {

    "Train": train,

    "Validation": validation,

    "Test": test,

}.items():

    print("=" * 60)

    print(name)

    print(
        "Image Features:",
        data["image_features"].shape,
    )

    print(
        "Text Features:",
        data["text_features"].shape,
    )

    print(
        "Metadata:",
        data["metadata"].shape,
    )

Train
Image Features: (257, 256)
Text Features: (257, 768)
Metadata: (257, 4)
Validation
Image Features: (56, 256)
Text Features: (56, 768)
Metadata: (56, 4)
Test
Image Features: (56, 256)
Text Features: (56, 768)
Metadata: (56, 4)


# Fusion Dataset

Create a PyTorch dataset that returns image features, text features,
and patient metadata.

In [7]:
class FusionDataset(Dataset):

    def __init__(

        self,

        image_features,

        text_features,

        metadata,

    ):

        self.image_features = image_features

        self.text_features = text_features

        self.metadata = metadata.reset_index(drop=True)

    def __len__(self):

        return len(self.metadata)

    def __getitem__(self, index):

        return {

            "image": torch.tensor(

                self.image_features[index],

                dtype=torch.float32,

            ),

            "text": torch.tensor(

                self.text_features[index],

                dtype=torch.float32,

            ),

            "patient_id":

                self.metadata.loc[index, "patient_id"],

            "metadata":

                self.metadata.iloc[index].to_dict(),

        }

In [8]:
train_dataset = FusionDataset(

    train["image_features"],

    train["text_features"],

    train["metadata"],

)

validation_dataset = FusionDataset(

    validation["image_features"],

    validation["text_features"],

    validation["metadata"],

)

test_dataset = FusionDataset(

    test["image_features"],

    test["text_features"],

    test["metadata"],

)

In [9]:
# 🔧 FIX: shuffle=True on train_loader.
# The original had shuffle=False on all three loaders — fine here since
# the encoder is not trained in this notebook (sanity check only), but
# any subsequent training notebook copying this pattern would train on
# the same order every epoch, which degrades generalization.
# Setting it correctly now so it is right when training is added.

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # 🔧 FIX: was False
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,      # correct: no shuffling for eval
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,      # correct: no shuffling for eval
)


In [10]:
batch = next(iter(train_loader))

print(batch["image"].shape)

print(batch["text"].shape)

print(len(batch["patient_id"]))

print(batch["metadata"]["patient_id"][:5])

torch.Size([16, 256])
torch.Size([16, 768])
16
['BraTS20_Training_007', 'BraTS20_Training_035', 'BraTS20_Training_025', 'BraTS20_Training_362', 'BraTS20_Training_264']


# Projection Heads

Project image features and text embeddings into a shared latent space
before multimodal fusion.

Both modalities are projected to the same embedding dimension to
facilitate feature interaction.
Cell 11

In [11]:
class ImageProjection(nn.Module):

    def __init__(self):

        super().__init__()

        self.projection = nn.Sequential(

            nn.Linear(256, 128),

            nn.LayerNorm(128),

            nn.GELU(),

            nn.Dropout(0.20),

        )

    def forward(self, x):

        return self.projection(x)

In [12]:
class TextProjection(nn.Module):

    def __init__(self):

        super().__init__()

        self.projection = nn.Sequential(

            nn.Linear(768, 512),

            nn.GELU(),

            nn.Dropout(0.30),

            nn.Linear(512, 256),

            nn.GELU(),

            nn.Dropout(0.20),

            nn.Linear(256, 128),

            nn.LayerNorm(128),

        )

    def forward(self, x):

        return self.projection(x)

# Fusion Block

Fuse projected image and text features into a unified multimodal
representation.

In [13]:
class FusionBlock(nn.Module):

    def __init__(self):

        super().__init__()

        self.fusion = nn.Sequential(

            nn.Linear(256, 256),

            nn.GELU(),

            nn.Dropout(0.30),

            nn.Linear(256, 256),

            nn.LayerNorm(256),

        )

    def forward(

        self,

        image,

        text,

    ):

        x = torch.cat(

            [

                image,

                text,

            ],

            dim=1,

        )

        return self.fusion(x)

# Multimodal Fusion Encoder

In [14]:
class FusionEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.image_projection = ImageProjection()

        self.text_projection = TextProjection()

        self.fusion_block = FusionBlock()

    def forward(

        self,

        image,

        text,

    ):

        image = self.image_projection(image)

        text = self.text_projection(text)

        representation = self.fusion_block(

            image,

            text,

        )

        return representation

In [15]:
encoder = FusionEncoder().to(device)

print(encoder)

FusionEncoder(
  (image_projection): ImageProjection(
    (projection): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.2, inplace=False)
    )
  )
  (text_projection): TextProjection(
    (projection): Sequential(
      (0): Linear(in_features=768, out_features=512, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=512, out_features=256, bias=True)
      (4): GELU(approximate='none')
      (5): Dropout(p=0.2, inplace=False)
      (6): Linear(in_features=256, out_features=128, bias=True)
      (7): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
  )
  (fusion_block): FusionBlock(
    (fusion): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.3, inplace=False)
      (3): Lin

# Sanity Check

In [16]:
batch = next(iter(train_loader))

image = batch["image"].to(device)

text = batch["text"].to(device)

representation = encoder(

    image,

    text,

)

print(representation.shape)

torch.Size([16, 256])


In [17]:
print(

    "Mean :",

    representation.mean().item(),

)

print(

    "Std :",

    representation.std().item(),

)

print(

    "NaN :",

    torch.isnan(

        representation

    ).sum().item(),

)

Mean : -6.05359673500061e-09
Std : 1.0000159740447998
NaN : 0


# Save Fusion Encoder

In [18]:
MODEL_DIR = Path(

    "/kaggle/working/models/fusion"

)

MODEL_DIR.mkdir(

    parents=True,

    exist_ok=True,

)

In [19]:
torch.save(
    {
        "model_state": encoder.state_dict(),
        "image_dim":   256,
        "text_dim":    768,
        "fusion_dim": 256,
        "output_dim":  256,
    },
    MODEL_DIR / "fusion_encoder_initial.pth",
)

print("Fusion encoder saved.")
print("  image_dim  :", 256)
print("  text_dim   :", 768)
print("  output_dim :", 256)


Fusion encoder saved.
  image_dim  : 256
  text_dim   : 768
  output_dim : 256
